# Chapter 02: Data and Sampling Distributions

**Primary Reference:**

Peter Bruce, Andrew Bruce, and Peter Gedeck,  
*Practical Statistics for Data Scientists: 50+ Essential Concepts Using R and Python (2nd Edition)*  
O'Reilly Media

## 1. Chapter Summary

**Data and sampling distributions** are fundamental concepts in statistical analysis that focus on **understanding how samples represent populations and how statistical estimates vary across repeated sampling processes**.

This chapter examines random sampling, sample bias, selection bias, regression to the mean, sampling distributions, the central limit theorem, standard error, bootstrap resampling, and confidence intervals.

Probability distributions including normal, Student’s t, binomial, chi‑square, F, Poisson, exponential, and Weibull are discussed, with Python implementations illustrating practical applications in real‑world data analysis.


## 2. Learning Objectives

This chapter aims to provide an understanding of:

- Explaining principles of random sampling and addressing sample bias.
- Analyzing causes of selection bias and describing regression to the mean.
- Understanding sampling distributions, applying the central limit theorem, and interpreting standard error.
- Applying bootstrap resampling for estimating variability and uncertainty.
- Constructing and interpreting confidence intervals using statistical methods.
- Conducting normal distribution analysis with z‑scores and Q‑Q plots.
- Examining long‑tailed distributions and evaluating implications for extreme values.
- Utilizing Student’s t‑distribution for estimating means with unknown variance.
- Applying binomial distribution for calculating probabilities in binary trials.
- Using chi‑square distribution for analyzing categorical data.
- Employing F‑distribution for variance comparisons and model evaluation.
- Modeling event counts, waiting times, and failure data with Poisson, exponential, and Weibull distributions.

## 3. Theoretical Explanation & Implementation

This section presents the theoretical concepts and practical implementations discussed in Chapter 2 through conceptual explanations, code reproduction, additional demonstrations, and analytical interpretations.

### 3.1 Random Sampling and Sample Bias

#### 3.1.1 Concept Explanation

**Population** refers to the entire defined dataset,  whereas a **sample** is a smaller subset selected to estimate and analyze the overall population characteristics.

**Random sampling** provides equal selection probability for each element in the dataset, while **stratified sampling** divides the population into subgroups and randomly chooses samples within those strata.

**Sample bias** occurs when selection systematically misrepresents a population, including self-selection in online reviews, so data quality and representativeness may matter more than sheer sample size.

#### Mathematical Representation

The sample mean summarizes the average value of the observed sample, while the sample proportion estimates the share of observations within a specific category:

$$
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i
$$

$$
\hat{p}_j=\frac{n_j}{n}
$$

| Symbol | Description |
| :--- | :--- |
| $\bar{x}$ | sample mean |
| $x_i$ | value of observation (i) |
| $n$ | total number of sample observation |
| $n_j$ | number of sample observations in category (j) |
| $\hat{p}_j$ | estimated proportion of category (j) |

#### 3.1.2 Code Reproduction

The reference material explains sampling concepts and the Literary Digest poll through text rather than Python code, this section utilizes synthetic polling data to compare sampling methods.

#### Additional Demonstration

The following demonstration simulates a voting population to compare **simple random sampling**, **proportionate stratified sampling**, and a **high-SES-only sample** that illustrates selection bias.

In [14]:
population = pd.DataFrame(
    {
        "Group": ["General"] * 700 + ["High-SES"] * 300,
        "Preference": (
            ["Roosevelt"] * 420
            + ["Landon"] * 280
            + ["Roosevelt"] * 90
            + ["Landon"] * 210
        ),
    }
)

def preference_percentages(data):
    return (
        data["Preference"]
        .value_counts(normalize=True)
        .reindex(["Roosevelt", "Landon"], fill_value=0) #output always Roosevelt or Landon, if one of them is missing, fill with 0
        .mul(100)
        .round(3) #3 numbers after the decimal point
    )

randomSample = population.sample(
    n=500, #take 500 samples from the population randomly
    random_state=42
)

stratifiedSample = (
    population.groupby("Group", group_keys=False)
    .sample(frac=0.5, random_state=42) #50% of each group is taken as a sample
)

biasedSample = (
    population.loc[population["Group"] == "High-SES"]
    .sample(n=200, random_state=42)
)

# Compare the population and the three sampling results
samples = {
    "Population": population,
    "Simple Random Sample": randomSample,
    "Stratified Sample": stratifiedSample,
    "High-SES-Only Sample": biasedSample,
}

summary_data = []

for label, data in samples.items():
    shares = preference_percentages(data)
    group_shares = (
        data["Group"]
        .value_counts(normalize=True)
        .mul(100)
        .round(1)
        .to_dict()
    )

    summary_data.append({
        "Sampling Method": label,
        "Sample Size (n)": len(data),
        "Roosevelt (%)": f"{shares['Roosevelt']:.3f}%",
        "Landon (%)": f"{shares['Landon']:.3f}%",
        "General Group (%)": f"{group_shares.get('General', 0.0):.1f}%",
        "High-SES Group (%)": f"{group_shares.get('High-SES', 0.0):.1f}%"
    })

comparisonTable = pd.DataFrame(summary_data)
display(comparisonTable.style.hide(axis="index"))

Sampling Method,Sample Size (n),Roosevelt (%),Landon (%),General Group (%),High-SES Group (%)
Population,1000,51.000%,49.000%,70.0%,30.0%
Simple Random Sample,500,50.400%,49.600%,73.4%,26.6%
Stratified Sample,500,53.200%,46.800%,70.0%,30.0%
High-SES-Only Sample,200,29.500%,70.500%,0.0%,100.0%


#### 3.1.3 Code Explanation

The initial dataset initializes a DataFrame of 1,000 hypothetical voters divided into **`General`** and **`High-SES`** groups, with preferences assigned to produce different voting patterns.

The `preference_percentages()` function converts raw candidate counts into percentages, while `sample()` extracts random observations without replacement and `groupby()` executes grouped operations for stratified sampling.

Furthermore, the execution loop aggregates voting percentages alongside group distributions to systematically compare the complete population against random, stratified, and biased sampling outcomes.

#### 3.1.4 Result Analysis & Interpretation

Within the total population, **Roosevelt receives 51.0%** preference compared to **Landon's 49.0%**, representing the true majority vote distribution of the overall benchmark dataset.

Furthermore, the **simple random sample** estimates 50.4% Roosevelt support, whereas the proportional stratified sample yields 53.2%, illustrating standard sampling variability across unbiased methods.

In contrast, the **High-SES-only sample** reports 29.5% Roosevelt and 70.5% Landon, reversing the actual majority sentiment due to systematic exclusion of the **`General`**  voter group.

### 3.2 Selection Bias

#### 3.2.1 Concept Explanation

**Selection bias** distorts statistical conclusions when sample observations are chosen systematically, particularly when patterns are discovered through repeated data exploration rather than testing predefined hypotheses.

Furthermore, **data snooping** exacerbates this issue through extensive pattern searching, while the **vast search effect** produces spurious findings due to evaluating numerous candidate models or predictors.

**Regression to the mean** explains why extreme results often move closer to average levels in later measurements, particularly when the first result includes unusually favorable luck.

#### Mathematical Representation

The probability that all ten independent coin tosses land heads equals the probability of heads multiplied across ten trials, assuming a fair coin:

$$
p_{10}=\left(\frac{1}{2}\right)^{10}=\frac{1}{1024}\approx 0.000977
$$

| Symbol | Description |
| :--- | :--- |
| $p_{10}$ | Probability that one participant obtains ten heads |
| $\frac{1}{2}$ | Probability of obtaining heads in one fair coin toss |
| $10$ | Number of coin tosses per participant |

For $N$ independent participants, the probability that at least one person obtains ten heads is one minus the probability that everyone fails:

$$
P(\text{at least one})=1-(1-p_{10})^N
$$

| Symbol | Description |
| :--- | :--- |
| $P(\text{at least one})$ | Probability that at least one participant obtains ten heads |
| $p_{10}$ | Probability that one participant obtains ten heads |
| $N$ | Total number of independent participants |

#### 3.2.2 Code Reproduction - Part 01

The reference material explains selection bias and the coin-toss thought experiment through text rather than Python code, this section utilizes synthetic simulations to demonstrate data snooping effects

#### Additional Demonstration

The following simulation models a 20,000-person stadium scenario, demonstrating how repeated coin tosses across a large population naturally produce extraordinary statistical outcomes purely by chance.

In [ ]:
np.random.seed(42) #set the seed so that the random numbers generated are reproducible
nPeople = 20000 #participants
nTosses = 10 #coin tosses

tossMatrix = np.random.randint(0, 2, size=(nPeople, nTosses)) #0 = inclusive, 2 = exclusive, so the random numbers generated are either 0 or 1

headsPerson = np.sum(tossMatrix, axis=1)
perfectStreaks = np.sum(headsPerson == nTosses) #participants who got 10 heads in a row

expectedStreaks = nPeople * (0.5 ** nTosses)
observedPercentage = (perfectStreaks / nPeople) * 100
expectedPercentage = (0.5 ** nTosses) * 100

summary = [
    {
        "Metric / Parameter": "Total Stadium Participants (n)",
        "Theoretical / Expected": f"{nPeople:,}",
        "Simulated Result": f"{nPeople:,}"
    },
    {
        "Metric / Parameter": "Consecutive Coin Tosses",
        "Theoretical / Expected": f"{nTosses}",
        "Simulated Result": f"{nTosses}"
    },
    {
        "Metric / Parameter": "Individual Success Probability",
        "Theoretical / Expected": f"{expectedPercentage:.3f}%",
        "Simulated Result": f"{observedPercentage:.3f}%"
    },
    {
        "Metric / Parameter": "Participants with 10 Perfect Heads",
        "Theoretical / Expected": f"{expectedStreaks:.3f}",
        "Simulated Result": f"{perfectStreaks}"
    }
]

stadiumTable = pd.DataFrame(summary)
display(stadiumTable.style.hide(axis="index"))

Metric / Parameter,Theoretical / Expected,Simulated Result
Total Stadium Participants (n),"20,000","20,000"
Consecutive Coin Tosses,10,10
Individual Success Probability,0.0977%,0.0800%
Participants with 10 Perfect Heads,19.53,16


#### 3.2.3 Code Explanation - Part 01

The code generates a reproducible matrix simulating ten coin tosses across 20,000 participants using `NumPy`,encoding heads as ones and tails as zeros.

By evaluating total heads per row, the script filters **`perfect streak`** counts and computes both theoretical expected values and empirical simulation percentages.

he aggregated metrics are organized into a structured pandas DataFrame and displayed cleanly without row indices to illustrate how repeated trials generate extreme outcomes.

#### 3.2.4 Result Analysis & Interpretation - Part 01

The simulation identifies 16 participants who achieved 10 consecutive heads, closely matching the theoretical expectation of 19.53 individuals for this stadium sample size.

Furthermore, the observed success rate of 0.080% closely aligns with the expected binomial probability of 0.097%, demonstrating minor random variation around the theoretical baseline.

Therefore, identifying a successful participant after observing all results illustrates the vast search effect, confirming that rare outcomes reflect random chance rather than individual skill.

In [ ]:
import numpy as np
import pandas as pd

# Set a fixed seed for reproducibility
rng = np.random.default_rng(seed=42)

# Define the simulation settings
n_people = 20_000
n_tosses = 10

# Simulate ten coin tosses for each participant
# 1 represents heads, while 0 represents tails
tosses = rng.integers(
    0,
    2,
    size=(n_people, n_tosses)
)

# Identify participants who obtain ten heads
all_heads = np.all(tosses == 1, axis=1)
n_all_heads = int(all_heads.sum())

# Calculate the theoretical probabilities
p_one_person = (0.5) ** n_tosses
p_at_least_one = 1 - (
    1 - p_one_person
) ** n_people

# Summarize the simulation results
summary = pd.DataFrame(
    {
        "Measure": [
            "Total participants",
            "Tosses per participant",
            "Participants with ten heads",
            "Observed percentage with ten heads",
            "Probability for one participant",
            "Probability of at least one success",
        ],
        "Value": [
            n_people,
            n_tosses,
            n_all_heads,
            f"{n_all_heads / n_people * 100:.3f}%",
            f"{p_one_person:.8f}",
            f"{p_at_least_one:.10f}",
        ],
    }
)

display(summary)

#### 3.2.2 Code Reproduction - Part 02

#### 3.2.3 Code Explanation - Part 02

#### 3.2.4 Result Analysis & Interpretation - Part 02

### 3.3 Sampling Distribution of a Statistic

#### 3.3.1 Concept Explanation

#### 3.3.2 Code Reproduction
#### 3.3.3 Code Explanation
#### 3.3.4 Result Analysis & Interpretation

### 3.4 The Bootstrap

#### 3.4.1 Concept Explanation

#### 3.4.2 Code Reproduction
#### 3.4.3 Code Explanation
#### 3.4.4 Result Analysis & Interpretation

### 3.5 Confidence Intervals

#### 3.5.1 Concept Explanation

#### 3.5.2 Code Reproduction
#### 3.5.3 Code Explanation
#### 3.5.4 Result Analysis & Interpretation

### 3.6 Normal Distribution

#### 3.6.1 Concept Explanation

#### 3.6.2 Code Reproduction
#### 3.6.3 Code Explanation
#### 3.6.4 Result Analysis & Interpretation

### 3.7 Long-Tailed Distributions

#### 3.7.1 Concept Explanation

#### 3.7.2 Code Reproduction
#### 3.7.3 Code Explanation
#### 3.7.4 Result Analysis & Interpretation

### 3.8 Student's t-Distribution

#### 3.8.1 Concept Explanation

#### 3.8.2 Code Reproduction
#### 3.8.3 Code Explanation
#### 3.8.4 Result Analysis & Interpretation

### 3.9 Binomial Distribution

#### 3.9.1 Concept Explanation

#### 3.9.2 Code Reproduction
#### 3.9.3 Code Explanation
#### 3.9.4 Result Analysis & Interpretation

### 3.10 Chi-Square Distribution

#### 3.10.1 Concept Explanation

#### 3.10.2 Code Reproduction
#### 3.10.3 Code Explanation
#### 3.10.4 Result Analysis & Interpretation

### 3.11 F-Distribution

#### 3.11.1 Concept Explanation

#### 3.11.2 Code Reproduction
#### 3.11.3 Code Explanation
#### 3.11.4 Result Analysis & Interpretation

### 3.12 Poisson and Related Distributions

#### 3.12.1 Concept Explanation

#### 3.12.2 Code Reproduction
#### 3.12.3 Code Explanation
#### 3.12.4 Result Analysis & Interpretation

## 4. Discussion & Conclusion

This chapter helped me to learn that **Exploratory Data Analysis (EDA)** is not only about calculating numbers but also about finding distributions, relationships, and patterns that explain the dataset. 

Applying **Python tools** such as `pandas`, `NumPy`, and `Seaborn` with diverse datasets showed how theoretical concepts can be applied in practice.

Even though textbook reproductions are limited, the exercise establishes an essential groundwork for systematic data exploration before progressing to advanced statistical models or machine‑learning approaches.